# Day 5: Pydantic Mastery - Validation, Serialization, and Schema Design

## Goal
Master Pydantic models for strict data validation, preventing invalid data from entering your API, and designing schemas that scale with your application.

## Why This Day Matters
Pydantic is unique to FastAPI/Python. It converts validation from **"something we hope to remember"** to **"guaranteed by the framework itself**." This prevents entire classes of bugs.

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from typing import Optional
from enum import Enum
import json

print("Pydantic: Automatic validation through Python type hints")

## PART 1: BASIC PYDANTIC MODELS

A Pydantic model is a Python class that defines the shape and validation rules for data.

In [ ]:
# EXAMPLE 1: Simple model with type hints
class User(BaseModel):
    id: int
    name: str
    email: str
    age: int
    is_active: bool = True  # Default value

# Creating instances
user1 = User(id=1, name="Safwan", email="safwan@example.com", age=25)
print(f"✓ User created: {user1.name}, age {user1.age}, active: {user1.is_active}")

# What validation happens automatically
try:
    invalid = User(id="not_a_number", name="Bob", email="bob@example.com", age=25)
except Exception as e:
    print(f"✓ Validation caught error: {type(e).__name__}")

# Type coercion
user2 = User(id="2", name="Alice", email="alice@example.com", age="30")
print(f"✓ Pydantic coerced string '2' to int 2, and '30' to int 30")

In [ ]:
# EXAMPLE 2: Optional fields
class UserProfile(BaseModel):
    name: str                    # Required
    email: str                   # Required
    bio: str | None = None       # Optional (defaults to None)
    website: str | None = None   # Optional (defaults to None)
    verified: bool = False       # Optional with different default

profile1 = UserProfile(name="Bob", email="bob@example.com")
print(f"✓ Created profile with optional fields: bio={profile1.bio}, website={profile1.website}")

profile2 = UserProfile(name="Alice", email="alice@example.com", bio="Developer", website="alice.dev")
print(f"✓ Same model with fields filled in: {profile2.bio}")

## PART 2: FIELD() FOR CONSTRAINTS

`Field()` adds validation rules and documentation.

In [ ]:
# EXAMPLE 1: String constraints
class ProductBasic(BaseModel):
    name: str = Field(
        ...,  # ... means required (no default)
        min_length=1,
        max_length=200,
        description="Product name (1-200 characters)"
    )
    description: str = Field(
        default="No description",
        min_length=0,
        max_length=5000,
        description="Detailed product description"
    )

try:
    p1 = ProductBasic(name="")  # Empty string
except Exception as e:
    print(f"✓ Min length validation works: {e}")

p2 = ProductBasic(name="iPhone 15")
print(f"✓ Created product: {p2.name}")

In [ ]:
# EXAMPLE 2: Numeric constraints
class PricedItem(BaseModel):
    price: float = Field(
        ...,
        gt=0,        # greater than (not equal)
        description="Must be positive"
    )
    discount_percent: int = Field(
        default=0,
        ge=0,        # greater than or equal
        le=100,      # less than or equal
        description="Discount 0-100%"
    )
    rating: float = Field(
        default=0.0,
        ge=0.0,
        le=5.0,
        description="Rating 0-5 stars"
    )

try:
    item1 = PricedItem(price=0)  # Not allowed
except Exception as e:
    print(f"✓ Numeric validation: price must be > 0")

item2 = PricedItem(price=49.99, discount_percent=10, rating=4.5)
print(f"✓ Price item valid: ${item2.price}, {item2.discount_percent}% off, {item2.rating}⭐")

## PART 3: REGEX PATTERNS

Validate strings against patterns.

In [ ]:
# EXAMPLE: Phone numbers, usernames, passwords
class UserAccount(BaseModel):
    username: str = Field(
        ...,
        min_length=3,
        max_length=50,
        pattern="^[a-zA-Z0-9_]+$",  # Only alphanumeric and underscore
        description="Username (alphanumeric + underscore only)"
    )
    phone: str = Field(
        ...,
        pattern="^\\d{10}$",  # Exactly 10 digits
        description="Phone number (10 digits)"
    )

try:
    user1 = UserAccount(username="user@name", phone="5551234567")  # @ not allowed
except Exception as e:
    print("✓ Pattern validation: username contains invalid character @")

user2 = UserAccount(username="user_name_123", phone="5551234567")
print(f"✓ Valid account: {user2.username}")

## PART 4: ENUMS FOR BOUNDED VALUES

Only allow specific predefined values.

In [ ]:
# EXAMPLE: Status, role, category enums
class OrderStatus(str, Enum):
    pending = "pending"
    processing = "processing"
    shipped = "shipped"
    delivered = "delivered"
    cancelled = "cancelled"

class UserRole(str, Enum):
    user = "user"
    moderator = "moderator"
    admin = "admin"

class Order(BaseModel):
    id: int
    status: OrderStatus = OrderStatus.pending  # Only specific values allowed
    user_role: UserRole = UserRole.user

order1 = Order(id=1)
print(f"✓ Order with default status: {order1.status.value}")

order2 = Order(id=2, status=OrderStatus.shipped)
print(f"✓ Order with shipped status: {order2.status.value}")

try:
    invalid = Order(id=3, status="invalid_status")  # Not in enum
except Exception as e:
    print(f"✓ Enum validation: only allowed values are {[s.value for s in OrderStatus]}")

## PART 5: NESTED MODELS

Models can contain other models for complex data structures.

In [ ]:
# EXAMPLE 1: Address nested in User
class Address(BaseModel):
    street: str
    city: str
    country: str
    zip_code: str = Field(..., pattern="^\\d{5}$")

class PersonalUser(BaseModel):
    name: str
    email: str
    address: Address  # Nested model

user = PersonalUser(
    name="Bob",
    email="bob@example.com",
    address={
        "street": "123 Main St",
        "city": "New York",
        "country": "USA",
        "zip_code": "10001"
    }
)

print(f"✓ User with nested address: {user.address.city}, {user.address.country}")
print(f"✓ Can access nested fields: {user.address.zip_code}")

In [ ]:
# EXAMPLE 2: Lists of nested models
class Comment(BaseModel):
    author: str
    text: str
    rating: int = Field(ge=1, le=5)

class Post(BaseModel):
    title: str
    content: str
    comments: list[Comment] = []  # List of comments
    tags: list[str] = []  # List of strings

post = Post(
    title="FastAPI Guide",
    content="Complete FastAPI tutorial",
    comments=[
        {"author": "Alice", "text": "Great post!", "rating": 5},
        {"author": "Bob", "text": "Very helpful", "rating": 4}
    ],
    tags=["fastapi", "python", "backend"]
)

print(f"✓ Post with {len(post.comments)} comments")
for comment in post.comments:
    print(f"  - {comment.author} ({comment.rating}⭐): {comment.text}")

## PART 6: CUSTOM VALIDATORS

Write custom validation logic for complex rules.

In [ ]:
# EXAMPLE 1: Field-level validator
class UserWithValidation(BaseModel):
    username: str
    email: str
    
    @field_validator("username")
    @classmethod
    def validate_username(cls, v: str) -> str:
        """
        Custom validation for username.
        - Must be at least 3 characters (already checked by Field)
        - Cannot be reserved names
        """
        reserved = {"admin", "root", "system", "test"}
        if v.lower() in reserved:
            raise ValueError(f"Username '{v}' is reserved")
        return v  # Return the cleaned value

try:
    user = UserWithValidation(username="admin", email="admin@example.com")
except Exception as e:
    print(f"✓ Custom validator caught reserved name: {e}")

user = UserWithValidation(username="myuser", email="user@example.com")
print(f"✓ Created user: {user.username}")

In [ ]:
# EXAMPLE 2: Model-level validator (cross-field validation)
class PasswordReset(BaseModel):
    password: str = Field(min_length=8)
    confirm_password: str
    
    @model_validator(mode="after")
    def passwords_match(self):
        """
        Validate that passwords match (needs both fields).
        Called after individual field validation.
        """
        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match")
        return self

try:
    reset = PasswordReset(password="Password123", confirm_password="Password456")
except Exception as e:
    print(f"✓ Model validator: {e}")

reset = PasswordReset(password="Password123", confirm_password="Password123")
print(f"✓ Valid password reset")

## PART 7: SEPARATE SCHEMAS (CREATE, UPDATE, READ)

Professional API design uses different schemas for different operations.

In [ ]:
# EXAMPLE: Three different schemas for one resource

class UserCreate(BaseModel):
    """Schema for creating a user. No ID, no timestamps."""
    name: str = Field(min_length=1, max_length=100)
    email: str
    password: str = Field(min_length=8)  # Hidden from response

class UserUpdate(BaseModel):
    """Schema for updating a user. All fields optional."""
    name: str | None = Field(default=None, min_length=1, max_length=100)
    email: str | None = None
    # Never include password in update - that's a separate "reset password" endpoint

class UserRead(BaseModel):
    """Schema for reading a user. No passwords, includes ID and timestamps."""
    id: int
    name: str
    email: str
    is_active: bool
    created_at: str  # Would be datetime in production
    
    # Models now have Config to work with ORM objects (we'll see in Day 6)
    class Config:
        from_attributes = True

print("✓ UserCreate: has password (for client input)")
print("✓ UserUpdate: all fields optional (for partial updates)")
print("✓ UserRead: no password (for API response)")

## PART 8: SERIALIZATION (CONVERTING TO JSON)

Pydantic makes converting to JSON and other formats easy.

In [ ]:
# EXAMPLE: Different serialization methods
user = UserCreate(
    name="Alice",
    email="alice@example.com",
    password="SecurePass1"
)

# Convert to dict
user_dict = user.model_dump()
print(f"✓ to dict: {type(user_dict)}")
print(f"  {user_dict}")

# Convert to JSON string
user_json = user.model_dump_json()
print(f"\n✓ to JSON: {type(user_json)}")
print(f"  {user_json}")

# Exclude certain fields
user_safe = user.model_dump(exclude={"password"})
print(f"\n✓ Exclude password: {user_safe}")

# Include only certain fields
user_minimal = user.model_dump(include={"name", "email"})
print(f"\n✓ Only name and email: {user_minimal}")

## PART 9: JSON SCHEMA FOR AUTO-DOCS

Pydantic automatically generates JSON schema for FastAPI /docs.

In [ ]:
# EXAMPLE: Schema generation
schema = UserCreate.model_json_schema()
print("✓ Generated JSON schema for docs:")
print(json.dumps(schema, indent=2)[:500] + "...")

## PART 10: PYDANTIC BEST PRACTICES

### ✅ DO:
1. Use type hints for every field
2. Separate Create, Update, Read schemas
3. Add descriptive error messages
4. Use `Field()` for constraints
5. Hide sensitive fields (`password`, `token`)
6. Use Enums for bounded values

### ❌ DON'T:
1. Use plain `dict` instead of Pydantic models
2. Expose database models in API responses
3. Skip validation on external input
4. Put secrets/passwords in response models
5. Use overly complex validators (keep it simple)

## Day 5 Summary

Pydantic automatically validates and converts data. You now understand:
✅ Basic models with type hints
✅ Constraints using `Field()`
✅ Enums for bounded values
✅ Nested models for complex structures
✅ Custom validators for business rules
✅ Separate schemas for Create/Update/Read
✅ Serialization to JSON and dicts
✅ Security (hiding sensitive fields)

**Next:** Day 6 - Database Integration with SQLAlchemy